# 🤖 Pocket-Agent — Full Training Pipeline

This notebook runs the **complete pipeline** from data generation to quantized inference:

1. 🔑 Set your Gemini API key (Cell 2)
2. 📦 Install dependencies
3. 📂 Clone the repository
4. 📊 Generate training data (2200+ examples)
5. 🏋️ Fine-tune SmolLM2-360M-Instruct with LoRA
6. ⚡ Quantize to INT8 for CPU inference
7. 🧪 Test inference + verify hard gates
8. 🚀 Launch Gradio chatbot demo

> **Runtime:** Make sure you're on a **T4 GPU** runtime: `Runtime > Change runtime type > T4 GPU`

---

## 🔑 Step 1: Set Your Gemini API Key

Get your free key at: **https://aistudio.google.com/apikeys**

Paste it in the cell below between the quotes.

In [ ]:
# ⬇️⬇️⬇️ PASTE YOUR GEMINI API KEY BELOW ⬇️⬇️⬇️
GEMINI_API_KEY = "PASTE_YOUR_KEY_HERE"
# ⬆️⬆️⬆️ PASTE YOUR GEMINI API KEY ABOVE ⬆️⬆️⬆️

import os
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

if GEMINI_API_KEY == "PASTE_YOUR_KEY_HERE":
    print("\u26a0\ufe0f WARNING: You haven't set your Gemini API key!")
    print("Get one free at: https://aistudio.google.com/apikeys")
    print("Data generation will use templates only (still works but less diverse).")
else:
    print(f"\u2705 Gemini API key set! (starts with {GEMINI_API_KEY[:8]}...)")

## 📦 Step 2: Install Dependencies

In [ ]:
!pip install -q transformers>=4.40.0 peft>=0.10.0 trl>=0.8.0 datasets>=2.18.0 accelerate>=0.28.0 gradio>=4.0.0 google-generativeai>=0.5.0
print("\n\u2705 All dependencies installed!")

## 📂 Step 3: Clone Repository

In [ ]:
import os

REPO_URL = "https://github.com/fastian-afk/pocket-agent.git"  # UPDATE THIS with your actual repo URL
REPO_DIR = "pocket-agent"

if os.path.exists(REPO_DIR):
    print(f"Directory {REPO_DIR} already exists. Pulling latest...")
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print(f"\n\u2705 Working directory: {os.getcwd()}")
!ls -la

## 📊 Step 4: Generate Training Data

This generates 2200+ ChatML-formatted training examples:
- 1800 single-turn tool calls (weather, calendar, convert, currency, sql)
- 350 multi-turn (context-dependent follow-ups)
- 350 adversarial (typos, Urdu/Spanish/Arabic code-switching)
- 200 refusals (chitchat, invalid tools)

If GEMINI_API_KEY is set, it adds ~130 extra Gemini-enhanced examples.

In [ ]:
!python generate_data.py

In [ ]:
# Verify data stats
import json

with open('train_data.jsonl', 'r', encoding='utf-8') as f:
    data = [json.loads(l) for l in f]

tools = {}
refusals = 0
multi_turn = 0
for ex in data:
    msgs = ex['messages']
    if len(msgs) > 3:
        multi_turn += 1
    for m in msgs:
        if m['role'] == 'assistant':
            if '<tool_call>' in m['content']:
                tc = json.loads(m['content'].split('<tool_call>')[1].split('</tool_call>')[0])
                tools[tc['tool']] = tools.get(tc['tool'], 0) + 1
            else:
                refusals += 1

print(f'\u2705 Total examples: {len(data)}')
print(f'   Multi-turn: {multi_turn}')
print(f'   Refusals: {refusals}')
print(f'   Tool calls:')
for t, c in sorted(tools.items()):
    print(f'     {t}: {c}')

assert len(data) >= 2000, f"\u274c Only {len(data)} examples! Need 2000+"
print(f'\n\u2705 Data generation passed! ({len(data)} >= 2000)')

## 🏋️ Step 5: Fine-Tune with LoRA

Fine-tunes `SmolLM2-360M-Instruct` with LoRA on the T4 GPU.
- LoRA: r=8, alpha=16, targets q_proj + v_proj
- 3 epochs, effective batch size 16

**Estimated time: ~10-20 minutes on T4**

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print()

!python train.py

In [ ]:
# Verify adapter was saved
import os
adapter_files = os.listdir('lora_adapter')
print(f"\u2705 LoRA adapter saved! Files: {adapter_files}")
adapter_size = sum(os.path.getsize(os.path.join('lora_adapter', f)) for f in adapter_files if os.path.isfile(os.path.join('lora_adapter', f)))
print(f"   Adapter size: {adapter_size / 1e6:.1f} MB")

## ⚡ Step 6: Quantize to INT8

Merges LoRA into the base model, then applies PyTorch dynamic INT8 quantization.

**Target: model file ≤ 500 MB (hard gate)**

In [ ]:
!python quantize.py

In [ ]:
# HARD GATE CHECK: Model size <= 500 MB
import os

model_path = 'quantized_model/model_quantized.pt'
model_size_mb = os.path.getsize(model_path) / 1e6

print(f"Quantized model size: {model_size_mb:.1f} MB")
print()

if model_size_mb <= 500:
    print(f"\u2705 HARD GATE PASSED: {model_size_mb:.1f} MB <= 500 MB")
else:
    print(f"\u274c HARD GATE FAILED: {model_size_mb:.1f} MB > 500 MB")

if model_size_mb <= 250:
    print(f"\ud83c\udfc6 BONUS: {model_size_mb:.1f} MB <= 250 MB (+10 points!)")

# Total directory size
total = sum(os.path.getsize(os.path.join('quantized_model', f)) for f in os.listdir('quantized_model') if os.path.isfile(os.path.join('quantized_model', f)))
print(f"\nTotal quantized_model/ directory: {total / 1e6:.1f} MB")

## 🧪 Step 7: Test Inference

Tests the `run()` function against sample queries and measures latency.

**Target: mean latency ≤ 200 ms/turn (hard gate)**

In [ ]:
!python inference.py

In [ ]:
# HARD GATE CHECK: No banned imports in inference.py
import ast

with open('inference.py', 'r') as f:
    tree = ast.parse(f.read())

banned = {'requests', 'urllib', 'http', 'socket', 'httpx', 'aiohttp'}
found_banned = []

for node in ast.walk(tree):
    if isinstance(node, ast.Import):
        for alias in node.names:
            if alias.name.split('.')[0] in banned:
                found_banned.append(alias.name)
    elif isinstance(node, ast.ImportFrom):
        if node.module and node.module.split('.')[0] in banned:
            found_banned.append(node.module)

if found_banned:
    print(f"\u274c BANNED IMPORTS FOUND: {found_banned}")
else:
    print("\u2705 HARD GATE PASSED: No banned network imports in inference.py")

# Check allowed imports
allowed = {'json', 're', 'torch', 'transformers', 'gradio', 'typing', 'os', 'time'}
imports = []
for node in ast.walk(tree):
    if isinstance(node, ast.Import):
        for alias in node.names:
            imports.append(alias.name.split('.')[0])
    elif isinstance(node, ast.ImportFrom):
        if node.module:
            imports.append(node.module.split('.')[0])

print(f"   Imports found: {set(imports)}")

## 🚀 Step 8: Launch Chatbot Demo

Launches a Gradio chatbot interface with a public URL.

**Try these prompts:**
- `What's the weather in Tokyo?`
- `Convert 100 miles to kilometers`
- `How much is 50 USD in EUR?`
- `Schedule 'Team Meeting' for 2025-04-20`
- `Tell me a joke` (should refuse)
- `Mujhe Lahore ka weather batao` (Urdu code-switch)

In [ ]:
!python app.py

## 🏁 All Done!

### Hard Gates Summary
| Gate | Status |
|---|---|
| Base model \u2264 2B params | \u2705 SmolLM2-360M (0.36B) |
| Quantized model \u2264 500 MB | Check Step 6 output |
| Mean inference \u2264 200 ms | Check Step 7 output |
| Zero train/test overlap | \u2705 SHA-256 verified |
| No network imports | Check Step 7 AST scan |
| Chatbot demo launches | \u2705 Gradio running |

### Bonus Points
| Bonus | Target |
|---|---|
| +10: Model \u2264 250 MB | Check Step 6 |
| +10: Beat GPT-4o-mini Slice C | Adversarial training data |
| +5: README error analysis | \u2705 Included |